# IT Operations Intelligence
## Fase 7-10: Modelado Predictivo y Evaluacion

**Objetivo:** Entrenar y evaluar multiples modelos de Machine Learning para predecir el incumplimiento de SLA en tickets de soporte tecnologico, comparando su rendimiento y seleccionando el mas adecuado.

In [14]:
import pandas as pd  # Libreria para manipulacion y analisis de datos
import numpy as np  # Libreria para operaciones numericas y matrices
import os  # Modulo para interactuar con el sistema operativo
import pickle  # Modulo para serializar/deserializar objetos Python
import matplotlib.pyplot as plt  # Libreria para crear graficos y visualizaciones
import seaborn as sns  # Libreria para visualizacion estadistica avanzada
from sklearn.model_selection import train_test_split, cross_val_score  # Dividir datos y validacion cruzada
from sklearn.preprocessing import StandardScaler  # Estandarizar caracteristicas numericas
from sklearn.linear_model import LogisticRegression  # Modelo de regresion logistica
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier  # Modelos ensemble
from sklearn.neural_network import MLPClassifier  # Red neuronal multicapa
from sklearn.metrics import (  # Metricas de evaluacion
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Tecnica para balancear clases mediante sobremuestreo
from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")  # Establecer estilo de fondo con cuadricula blanca
pd.set_option("display.max_columns", None)  # Mostrar todas las columnas sin truncar
ruta_graficos = "../reports/graphics"  # Ruta donde se guardaran los graficos generados
os.makedirs(ruta_graficos, exist_ok=True)  # Crear el directorio si no existe


In [15]:
# Cargar el dataset transformado desde el archivo CSV procesado
df = pd.read_csv("../data/processed/incident_event_log_transformado.csv")

# Mostrar las primeras 5 filas para inspeccionar el DataFrame
df.head()


##### Analisis de los datos cargados
Se observan las primeras filas del dataset transformado. Contiene 48 columnas con informacion de incidencias IT, incluyendo variables temporales (open_hour, open_dayofweek), metricas de severidad (priority_score, impact_score) y la variable objetivo made_sla_num que indica si se incumplio el SLA (1) o no (0). Los datos incluyen tanto columnas categoricas como numericas listas para el modelado.

In [16]:
# Obtener las dimensiones del DataFrame (filas, columnas)
df.shape


##### Analisis de dimensiones del dataset
El dataset contiene 141,712 registros y 48 columnas. Esto representa un volumen significativo de datos para entrenar modelos predictivos, lo cual es favorable para obtener resultados robustos. La alta dimensionalidad requiere una seleccion cuidadosa de variables predictoras.

In [17]:
# Mostrar información detallada del DataFrame: tipos de datos y valores no nulos
df.info()


##### Analisis de tipos de datos y valores nulos
El DataFrame contiene 4 columnas booleanas, 2 float, 12 enteros y 30 de tipo string (objeto). Las columnas 'sys_created_at', 'resolved_at' y 'resolution_time_hours' presentan valores nulos (88,636 y 138,571 no nulos respectivamente), lo que indica que algunos tickets no tienen resolucion registrada. Esto requerira imputacion para el modelado.

In [18]:
# Mostrar estadísticas descriptivas de las columnas numéricas
df.describe()


##### Analisis estadistico descriptivo
Las estadisticas revelan informacion clave: el promedio de reassignment_count es ~1.1, el tiempo de resolucion promedio es ~270 horas con una alta desviacion estandar (651 horas), indicando gran variabilidad. La variable objetivo 'made_sla_num' tiene media de 0.935, confirmando que ~93.5% de los tickets cumplen SLA. 'problematic_ticket' tiene media de 0.148, indicando que ~14.8% de los tickets son problematicos.

In [19]:
# Calcular el total de valores nulos en todo el DataFrame
df.isnull().sum().sum()


##### Analisis de valores nulos totales
Existen 59,358 valores nulos en total en el dataset. Esto se debe principalmente a que columnas como 'resolution_time_hours' y 'sys_created_at' tienen registros incompletos para tickets que nunca fueron resueltos. Sera necesario imputar estos valores antes del entrenamiento.

In [20]:
# Contar el número de filas duplicadas en el DataFrame
df.duplicated().sum()


##### Analisis de duplicados
No se encontraron filas duplicadas en el dataset (0 duplicados). Esto indica que los datos han sido correctamente desduplicados durante la fase de preprocesamiento, lo cual es adecuado para el modelado.

### Fase 7: Preparacion para Machine Learning

In [21]:
# Definir la variable objetivo: made_sla_num (1 = No cumple SLA, 0 = Sí cumple SLA)
target = "made_sla_num"

# Mostrar título de la distribución
print("Distribución de la variable objetivo:")

# Mostrar conteo de cada clase (0 y 1)
print(df[target].value_counts())

# Línea en blanco para separar secciones
print()

# Mostrar título del porcentaje
print("Porcentaje:")

# Mostrar porcentaje de cada clase con 2 decimales
print(round(df[target].value_counts(normalize=True) * 100, 2))


##### Analisis de distribucion de la variable objetivo
Existe un fuerte desbalance de clases: 132,497 tickets cumplen SLA (93.5%) frente a solo 9,215 que no cumplen (6.5%). Este desbalance es critico y requerira tecnicas de balanceo como SMOTE para evitar que los modelos ignoren la clase minoritaria (incumplimiento de SLA), que es precisamente la mas importante de predecir.

In [22]:
# Crear figura con tamaño de 6x4 pulgadas
plt.figure(figsize=(6, 4))

# Crear gráfico de barras mostrando frecuencia de cada clase
sns.countplot(data=df, x=target)

# Establecer título del gráfico
plt.title("Distribución de Clases - Cumplimiento de SLA")

# Guardar gráfico en formato PNG a 300 dpi
plt.savefig(f"{ruta_graficos}/Nb04_01_distribucion_clases.png", dpi=300, bbox_inches="tight")

# Mostrar el gráfico en pantalla
plt.show()


##### Analisis del grafico de distribucion de clases
El grafico de barras confirma visualmente el severo desbalance entre clases. La clase 0 (Cumple SLA) domina ampliamente mientras que la clase 1 (No Cumple SLA) es minoritaria. Este desbalance justifica el uso de SMOTE y class_weight='balanced' en los modelos para evitar predicciones sesgadas hacia la clase mayoritaria.

In [23]:
# Asignar la columna objetivo a la variable y
y = df[target]

# Definir columnas a excluir por ser identificadores o datos no predictivos
excluir = [target, "made_sla", "number", "caller_id", "opened_by",
           "sys_created_by", "sys_updated_by", "assigned_to", "resolved_by",
           "opened_at", "resolved_at", "closed_at", "sys_created_at", "sys_updated_at",
           "incident_state", "closed_code", "problem_id", "rfc", "vendor", "caused_by"]

# Eliminar las columnas excluidas y crear la matriz de predictores X
X = df.drop(columns=[c for c in excluir if c in df.columns], errors="ignore")

# Seleccionar únicamente las columnas numéricas para el modelado
X = X.select_dtypes(include=np.number)

# Iterar sobre cada columna de X para imputar valores nulos
for col in X.columns:

    # Si la columna tiene al menos un valor nulo, rellenar con la mediana
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())

# Listar las variables seleccionadas
print(f"Variables predictoras: {X.columns.tolist()}")

# Verificar que no queden valores nulos
print(f"Total NaN en X: {X.isnull().sum().sum()}")

# Mostrar dimensiones de X e y
print(f"Dimensiones X: {X.shape}, y: {y.shape}")


##### Analisis de las variables predictoras seleccionadas
Se seleccionaron 13 variables numericas predictoras: reassignment_count, reopen_count, sys_mod_count, open_hour, open_dayofweek, open_month, open_year, resolution_time_hours, priority_score, impact_score, urgency_score, criticality_score y problematic_ticket. No quedan valores nulos en X tras la imputacion con mediana. La dimension final es 141,712 filas por 13 columnas.

In [24]:
# Separar datos en entrenamiento (80%) y prueba (20%) manteniendo proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Mostrar dimensiones de entrenamiento y prueba
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# Mostrar distribución de clases en entrenamiento
print(f"y_train distribución: {y_train.value_counts().to_dict()}")

# Mostrar distribución de clases en prueba
print(f"y_test distribución: {y_test.value_counts().to_dict()}")


##### Analisis de la division entrenamiento/prueba
Se obtuvieron 113,369 muestras para entrenamiento y 28,343 para prueba, manteniendo la proporcion de clases gracias a stratify=y. La distribucion de clases en ambos conjuntos refleja fielmente el desbalance original (~93.5% clase 0, ~6.5% clase 1), asegurando una evaluacion representativa.

In [25]:
# Crear objeto StandardScaler para estandarizar características numéricas
scaler = StandardScaler()

# Identificar columnas numéricas
numeric_cols = X_train.select_dtypes(include=np.number).columns

# Aplicar fit_transform en entrenamiento para calcular media y desviación
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index
)

# Aplicar transform en prueba usando parámetros de entrenamiento
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index
)

# Confirmación del proceso de escalado
print("Escalado de características completado.")

# Mostrar dimensiones del conjunto escalado de entrenamiento
print(f"X_train_scaled: {X_train_scaled.shape}")

# Mostrar dimensiones del conjunto escalado de prueba
print(f"X_test_scaled: {X_test_scaled.shape}")


##### Analisis del escalado de caracteristicas
Se aplico estandarizacion (StandardScaler) a las 13 variables numericas, transformandolas para que tengan media 0 y desviacion estandar 1. Esto es esencial para modelos como Regresion Logistica y Red Neuronal MLP que son sensibles a la escala de las variables. Random Forest y Gradient Boosting no requieren escalado por ser basados en arboles.

### Fase 8: Modelado Predictivo

#### Modelo 1: Regresion Logistica

In [26]:
# Crear objeto SMOTE con semilla fija para balancear clases
smote = SMOTE(random_state=42)

# Balancear clases en el conjunto de entrenamiento
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Mostrar distribución tras balanceo
print(f"Después de SMOTE: {y_train_resampled.value_counts().to_dict()}")


##### Analisis del balanceo con SMOTE
Tras aplicar SMOTE, ambas clases quedan igualadas con 105,997 muestras cada una. Esto elimina el desbalance y permite que el modelo aprenda adecuadamente de la clase minoritaria (incumplimiento de SLA), evitando el sesgo hacia la clase mayoritaria.

In [27]:
# Inicializar modelo de regresión logística con pesos balanceados
modelo_lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Entrenar el modelo con datos balanceados por SMOTE
modelo_lr.fit(X_train_resampled, y_train_resampled)

# Confirmación de entrenamiento
print("Modelo de Regresión Logística entrenado.")

# Mostrar número de coeficientes del modelo (uno por variable)
print(f"Coeficientes: {len(modelo_lr.coef_[0])}")

# Mostrar valor del intercepto (sesgo) del modelo
print(f"Intercepto: {modelo_lr.intercept_[0]:.4f}")


##### Analisis del entrenamiento de Regresion Logistica
El modelo de Regresion Logistica se entreno exitosamente con 13 coeficientes (uno por variable predictora) y un intercepto de 0.9232. Los coeficientes permitiran interpretar la direccion e intensidad del impacto de cada variable en la probabilidad de incumplimiento de SLA.

In [28]:
# Predecir clases en el conjunto de prueba
y_pred_lr = modelo_lr.predict(X_test_scaled)

# Obtener probabilidades de la clase positiva (1)
y_proba_lr = modelo_lr.predict_proba(X_test_scaled)[:, 1]

# Título de la evaluación
print("Evaluación - Regresión Logística:")

# Mostrar reporte de clasificación detallado
print(classification_report(y_test, y_pred_lr))


##### Analisis del rendimiento - Regresion Logistica
La Regresion Logistica muestra un recall del 75% para la clase 0 (incumplimiento SLA), lo cual es aceptable para detectar tickets en riesgo. Sin embargo, la precision es baja (21%) para esta clase, generando muchos falsos positivos. El F1-Score general ponderado es 0.85 y el accuracy global es 80%. El ROC-AUC de 0.8526 indica discriminacion moderada entre clases.

#### Modelo 2: Random Forest

In [29]:
# Inicializar modelo Random Forest con 200 árboles y pesos balanceados
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

# Entrenar el modelo con datos originales (sin escalar)
modelo_rf.fit(X_train, y_train)

# Confirmación de entrenamiento
print("Modelo Random Forest entrenado.")

# Mostrar cantidad de árboles creados en el bosque
print(f"Número de árboles: {len(modelo_rf.estimators_)}")


##### Analisis del entrenamiento de Random Forest
El Random Forest se entreno con 200 arboles de decision utilizando los datos originales sin escalar (los modelos basados en arboles no requieren estandarizacion) y con class_weight='balanced' para manejar el desbalance de clases sin necesidad de SMOTE.

In [30]:
# Predecir clases en el conjunto de prueba
y_pred_rf = modelo_rf.predict(X_test)

# Obtener probabilidades de la clase positiva
y_proba_rf = modelo_rf.predict_proba(X_test)[:, 1]

# Título de la evaluación
print("Evaluación - Random Forest:")

# Mostrar reporte de clasificación detallado
print(classification_report(y_test, y_pred_rf, zero_division=0))


##### Analisis del rendimiento - Random Forest
Random Forest mejora el accuracy al 90%, con un recall del 94% para la clase mayoritaria. Sin embargo, el recall para la clase minoritaria (0) es solo del 29%, y la precision del 25%. El F1-Score para clase 0 es bajo (0.27), indicando que el modelo tiene dificultades para identificar correctamente los tickets que incumplen SLA, aunque el accuracy general es superior a la Regresion Logistica.

#### Modelo 3: Gradient Boosting

In [31]:
# Inicializar modelo Gradient Boosting con 200 árboles y profundidad máxima 5
modelo_gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

# Entrenar el modelo con datos de entrenamiento
modelo_gb.fit(X_train, y_train)

# Confirmación de entrenamiento
print("Modelo Gradient Boosting entrenado.")


##### Analisis del entrenamiento de Gradient Boosting
El modelo Gradient Boosting se entreno con 200 estimadores y profundidad maxima de 5. A diferencia de Random Forest, Gradient Boosting construye arboles secuencialmente donde cada nuevo arbol corrige los errores del anterior, lo que puede lograr mayor precision pero con mayor riesgo de sobreajuste.

In [32]:
# Predecir clases en el conjunto de prueba
y_pred_gb = modelo_gb.predict(X_test)

# Obtener probabilidades de la clase positiva
y_proba_gb = modelo_gb.predict_proba(X_test)[:, 1]

# Título de la evaluación
print("Evaluación - Gradient Boosting:")

# Mostrar reporte de clasificación detallado
print(classification_report(y_test, y_pred_gb))


##### Analisis del rendimiento - Gradient Boosting
Gradient Boosting alcanza un accuracy del 93.5% y un recall excepcional del 100% para la clase 1. Sin embargo, el recall para la clase 0 es solo del 3%, lo que significa que apenas detecta los verdaderos incumplimientos de SLA. La precision para clase 0 es del 46%, pero el F1-Score de 0.06 refleja un pobre desempeno en la clase minoritaria. Esto indica sobreajuste a la clase mayoritaria.

#### Modelo 4: Red Neuronal MLP

In [33]:
# Inicializar clasificador MLP con tres capas ocultas (64, 32, 16 neuronas)
modelo_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

# Entrenar con datos escalados (requisito del MLP por ser basado en gradiente)
modelo_mlp.fit(X_train_scaled, y_train)

# Mostrar iteraciones reales usadas hasta convergencia
print(f"Modelo MLP entrenado. Iteraciones: {modelo_mlp.n_iter_}")


##### Analisis del entrenamiento de la Red Neuronal MLP
La red neuronal con arquitectura de 3 capas ocultas (64-32-16 neuronas) se entreno en 258 iteraciones (convergio antes del maximo de 500). Al ser un modelo basado en gradiente, requiere datos escalados (X_train_scaled). La arquitectura relativamente profunda permite capturar relaciones no lineales complejas entre las variables predictoras.

In [34]:
# Predecir clases en el conjunto de prueba escalado
y_pred_mlp = modelo_mlp.predict(X_test_scaled)

# Obtener probabilidades de la clase positiva
y_proba_mlp = modelo_mlp.predict_proba(X_test_scaled)[:, 1]

# Título de la evaluación
print("Evaluación - Red Neuronal MLP:")

# Mostrar reporte de clasificación detallado
print(classification_report(y_test, y_pred_mlp))


##### Analisis del rendimiento - Red Neuronal MLP
El MLP muestra un rendimiento similar a Gradient Boosting: accuracy del 93.4%, recall del 99% para clase 1, pero solo 6% de recall para clase 0. La precision para clase 0 es del 44% y F1-Score de 0.10. La red neuronal tambien sufre del desbalance de clases, prediciendo mayoritariamente la clase mayoritaria.

### Fase 9: Comparacion y Evaluacion de Modelos

In [35]:
# Crear DataFrame con las métricas de rendimiento de todos los modelos
resultados = pd.DataFrame({
    "Modelo": [
        "Regresión Logística",
        "Random Forest",
        "Gradient Boosting",
        "Red Neuronal MLP"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_mlp)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr, zero_division=0),
        precision_score(y_test, y_pred_rf, zero_division=0),
        precision_score(y_test, y_pred_gb, zero_division=0),
        precision_score(y_test, y_pred_mlp, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_gb, zero_division=0),
        recall_score(y_test, y_pred_mlp, zero_division=0)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_gb, zero_division=0),
        f1_score(y_test, y_pred_mlp, zero_division=0)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_gb),
        roc_auc_score(y_test, y_proba_mlp)
    ]
})

# Mostrar título de la comparación
print("Comparación de Modelos:")

# Mostrar tabla redondeada a 4 decimales
print(resultados.round(4))

# Línea en blanco
print()

# Mostrar título del mejor modelo
print("Mejor modelo por F1-Score:")

# Mostrar fila del modelo con mejor F1-Score
print(resultados.loc[resultados["F1-Score"].idxmax()])


##### Analisis comparativo de modelos
La tabla comparativa revela:
- **Gradient Boosting** obtiene el mejor F1-Score (0.9661) y ROC-AUC (0.9056), seguido muy de cerca por Red Neuronal MLP (F1: 0.9657).
- **Regresion Logistica** tiene el menor accuracy (80.3%) pero el mejor balance entre recall de clase 0 (75%) y clase 1 (81%), siendo el modelo mas equilibrado.
- **Random Forest** logra buen accuracy (89.7%) pero bajo desempeno en la clase minoritaria.
- Gradient Boosting y MLP tienen alta accuracy (>93%) pero casi no detectan la clase 0 (incumplimiento SLA), lo que los hace poco practicos para el objetivo de negocio de identificar tickets en riesgo.

### Fase 10: Interpretacion de Resultados

In [36]:
# Obtener el índice del modelo con mayor F1-Score
mejor_idx = resultados["F1-Score"].idxmax()

# Obtener el nombre del mejor modelo
mejor_modelo_nombre = resultados.loc[mejor_idx, "Modelo"]

# Obtener el valor del mejor F1-Score
mejor_f1 = resultados.loc[mejor_idx, "F1-Score"]

# Mostrar el mejor modelo según F1-Score
print(f"Mejor modelo: {mejor_modelo_nombre} (F1-Score: {mejor_f1:.4f})")

# Asignar el modelo seleccionado como modelo final según corresponda
if mejor_modelo_nombre == "Random Forest":
    modelo_final = modelo_rf
    y_pred_final = y_pred_rf
    y_proba_final = y_proba_rf

elif mejor_modelo_nombre == "Gradient Boosting":
    modelo_final = modelo_gb
    y_pred_final = y_pred_gb
    y_proba_final = y_proba_gb

elif mejor_modelo_nombre == "Red Neuronal MLP":
    modelo_final = modelo_mlp
    y_pred_final = y_pred_mlp
    y_proba_final = y_proba_mlp

else:
    modelo_final = modelo_lr
    y_pred_final = y_pred_lr
    y_proba_final = y_proba_lr

# Confirmar modelo seleccionado como final
print(f"Modelo final seleccionado: {mejor_modelo_nombre}")


##### Analisis del modelo seleccionado
El modelo **Gradient Boosting** fue seleccionado como el mejor segun el F1-Score (0.9661). Si bien tiene limitaciones en la deteccion de la clase minoritaria, su alto rendimiento general y ROC-AUC de 0.9056 lo posicionan como el mas robusto. Para mejorar la deteccion de incumplimientos, se podrian ajustar los umbrales de decision o aplicar tecnicas adicionales de balanceo.

In [37]:
# Calcular matriz de confusión con valores reales y predichos
cm = confusion_matrix(y_test, y_pred_final)

# Título de la matriz
print("Matriz de Confusión:")

# Mostrar matriz de confusión numérica
print(cm)

# Crear objeto para visualizar la matriz de confusión con etiquetas descriptivas
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Cumple SLA", "No Cumple SLA"]
)

# Graficar la matriz de confusión
disp.plot()

# Establecer título del gráfico con nombre del modelo
plt.title(f"Matriz de Confusión - {mejor_modelo_nombre}")

# Guardar gráfico en PNG a 300 dpi
plt.savefig(f"{ruta_graficos}/Nb04_02_matriz_confusion.png", dpi=300, bbox_inches="tight")

# Mostrar la matriz de confusión en pantalla
plt.show()


##### Analisis de la matriz de confusion
La matriz de confusion del modelo Gradient Boosting muestra:
- **Verdaderos negativos (Cumple SLA correctos):** 64
- **Falsos positivos (falsas alarmas):** 1,779 (incidencias que cumplieron SLA pero el modelo predijo que no)
- **Falsos negativos (incumplimientos no detectados):** 75
- **Verdaderos positivos (incumplimientos correctos):** 26,425
El modelo detecta la gran mayoria de los casos de cumplimiento (26,425 de 26,500), pero genera muchos falsos positivos (1,779), lo que indica una tendencia a sobre-predecir incumplimientos.

In [38]:
# Crear figura de 12x6 pulgadas para la comparación de métricas
plt.figure(figsize=(12, 6))

# Definir las métricas a graficar
metricas_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

# Transformar resultados a formato largo para seaborn
resultados_melted = resultados.melt(id_vars=["Modelo"], value_vars=metricas_plot,
                                     var_name="Métrica", value_name="Valor")

# Crear gráfico de barras agrupadas por modelo y métrica
sns.barplot(data=resultados_melted, x="Modelo", y="Valor", hue="Métrica")

# Establecer título del gráfico
plt.title("Comparación de Métricas entre Modelos")

# Rotar etiquetas del eje X para mejor legibilidad
plt.xticks(rotation=15)

# Posicionar leyenda fuera del gráfico
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

# Ajustar layout para evitar superposiciones
plt.tight_layout()

# Guardar gráfico en PNG a 300 dpi
plt.savefig(f"{ruta_graficos}/Nb04_03_comparacion_modelos.png", dpi=300, bbox_inches="tight")

# Mostrar el gráfico de comparación
plt.show()


##### Analisis del grafico de comparacion de modelos
El grafico de barras agrupadas permite visualizar las diferencias entre modelos para cada metrica. Gradient Boosting y MLP dominan en Accuracy, Recall y F1-Score, mientras que Regresion Logistica muestra el ROC-AUC mas bajo. Random Forest tiene un rendimiento intermedio pero mas equilibrado. La Regresion Logistica destaca por tener el mejor recall para la clase minoritaria (visible en el analisis detallado).

### Importancia de Variables (mejor modelo basado en arboles)

In [39]:
# Verificar si el modelo tiene el atributo feature_importances_
if hasattr(modelo_final, "feature_importances_"):

    # Crear DataFrame con nombres de variables y sus importancias ordenadas
    importancia = pd.DataFrame({
        "Variable": X.columns,
        "Importancia": modelo_final.feature_importances_
    }).sort_values("Importancia", ascending=False)

    # Mostrar título del ranking
    print("Top 10 variables más importantes:")

    # Mostrar las 10 variables más importantes
    print(importancia.head(10))

    # Crear figura de 10x6 pulgadas
    plt.figure(figsize=(10, 6))

    # Crear gráfico de barras horizontal con las 15 variables más importantes
    sns.barplot(data=importancia.head(15), x="Importancia", y="Variable")

    # Establecer título con nombre del modelo
    plt.title(f"Importancia de Variables - {mejor_modelo_nombre}")

    # Ajustar layout
    plt.tight_layout()

    # Guardar gráfico en PNG
    plt.savefig(f"{ruta_graficos}/Nb04_04_importancia_variables.png", dpi=300, bbox_inches="tight")

    # Mostrar gráfico de importancia
    plt.show()

else:
    # Mostrar mensaje si el modelo no proporciona importancia de variables
    print("El modelo seleccionado no proporciona importancia de variables.")


##### Analisis de importancia de variables
Las 3 variables mas importantes segun el modelo Gradient Boosting son:
1. **sys_mod_count** (53.5%): Numero de modificaciones del ticket, con diferencia abrumadora sobre las demas.
2. **resolution_time_hours** (23.4%): Tiempo de resolucion en horas.
3. **reassignment_count** (6.9%): Numero de reasignaciones del ticket.
Esto sugiere que los tickets que requieren muchas modificaciones y reasignaciones, o tienen largos tiempos de resolucion, son los que mas probablemente incumpliran el SLA. Estas variables son clave para la gestion proactiva de incidencias.

### Exportacion de Resultados

In [40]:
# Crear directorio final si no existe
os.makedirs("../data/final", exist_ok=True)

# Exportar comparación de modelos a CSV
resultados.to_csv("../data/final/comparacion_modelos.csv", index=False)

# Confirmación de exportación
print("Comparación de modelos guardada.")


In [41]:
# Copiar predictores del conjunto de prueba
predicciones = X_test.copy()

# Añadir columna con valores reales
predicciones["Valor_Real"] = y_test.values

# Añadir columna con predicciones del modelo
predicciones["Predicción"] = y_pred_final

# Añadir columna con probabilidades predichas
predicciones["Probabilidad"] = y_proba_final

# Filtrar predicciones positivas (tickets en riesgo de incumplimiento)
incidentes_riesgo = predicciones[predicciones["Predicción"] == 1]

# Mostrar cantidad de tickets en riesgo
print(f"Tickets en riesgo de incumplimiento de SLA: {len(incidentes_riesgo)}")

# Guardar todas las predicciones en CSV
predicciones.to_csv("../data/final/predicciones_incidencias.csv", index=False)

# Guardar solo tickets en riesgo en CSV
incidentes_riesgo.to_csv("../data/final/incidentes_riesgo_sla.csv", index=False)

# Confirmación de exportación
print("Predicciones e incidencias de riesgo guardadas.")


In [42]:
# Crear directorio de modelos si no existe
os.makedirs("../models", exist_ok=True)

# Abrir archivo binario para escritura y serializar el modelo entrenado
with open("../models/modelo_sla.pkl", "wb") as f:
    pickle.dump(modelo_final, f)

# Confirmación de guardado
print("Modelo guardado correctamente como 'modelo_sla.pkl'")
